# Notebook 03 — Data Preprocessing and Feature Engineering

## Objective

This notebook transforms the feature set retained following the feature-audit and target-leakage analysis into a modelling-ready dataset for subsequent credit-risk experiments.

The preprocessing stage preserves the distinction between structured borrower information and the borrower-written loan description used as the textual modality.

### Objectives

1. Load the audited feature definitions produced by Notebook 02.
2. Construct the resolved binary modelling population.
3. Create the binary default target.
4. Separate structured predictors from the textual `desc` feature.
5. Perform required feature engineering.
6. Define appropriate missing-value treatment.
7. Prepare structured features for modelling.
8. Preserve the text feature for subsequent text-representation experiments.
9. Prevent data leakage by ensuring that preprocessing parameters are learned from training data only.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Credit_Risk_Thesis"
)

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

NB2_OUTPUT_DIR = PROCESSED_DATA_DIR / "notebook_02"
NB3_OUTPUT_DIR = PROCESSED_DATA_DIR / "notebook_03"

NB3_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)
print("Notebook 2 inputs:", NB2_OUTPUT_DIR)
print("Notebook 3 outputs:", NB3_OUTPUT_DIR)

Project root: /content/drive/MyDrive/Credit_Risk_Thesis
Notebook 2 inputs: /content/drive/MyDrive/Credit_Risk_Thesis/data/processed/notebook_02
Notebook 3 outputs: /content/drive/MyDrive/Credit_Risk_Thesis/data/processed/notebook_03


## 1. Load Audited Feature Definitions

Notebook 02 established the candidate information set through feature auditing and target-leakage assessment. The resulting feature inventory and retained candidate-feature list are loaded directly here so that feature-selection decisions are not repeated during preprocessing.

In [4]:
FEATURE_INVENTORY_FILE = (
    NB2_OUTPUT_DIR / "feature_inventory_final.csv"
)

CANDIDATE_FEATURE_FILE = (
    NB2_OUTPUT_DIR / "final_candidate_features.csv"
)

print("Feature inventory exists:", FEATURE_INVENTORY_FILE.exists())
print("Candidate feature list exists:", CANDIDATE_FEATURE_FILE.exists())

Feature inventory exists: True
Candidate feature list exists: True


In [5]:
feature_inventory = pd.read_csv(
    FEATURE_INVENTORY_FILE
)

candidate_features_df = pd.read_csv(
    CANDIDATE_FEATURE_FILE
)

candidate_features = (
    candidate_features_df["feature"]
    .tolist()
)

print("Feature inventory shape:", feature_inventory.shape)
print("Candidate feature count:", len(candidate_features))

Feature inventory shape: (151, 6)
Candidate feature count: 64


In [6]:
assert len(feature_inventory) == 151
assert len(candidate_features) == 64

assert "loan_status" not in candidate_features
assert "desc" in candidate_features
assert "issue_d" not in candidate_features

print("Notebook 02 handoff validation passed.")

Notebook 02 handoff validation passed.


## 2. Load Required Raw Data

Only the variables required for the preprocessing stage are loaded from the original Lending Club dataset. This includes the 64 retained candidate source features, the loan-status variable required to construct the binary target, and `issue_d`, which is retained only as an auxiliary temporal variable for feature engineering.

Loading only the required columns reduces memory consumption while maintaining a clear separation between retained predictors, the outcome variable, and auxiliary transformation variables.

In [7]:
RAW_DATA_FILE = (
    RAW_DATA_DIR / "accepted_2007_to_2018Q4.csv"
)

print("Raw dataset exists:", RAW_DATA_FILE.exists())
print("Raw dataset:", RAW_DATA_FILE)

Raw dataset exists: True
Raw dataset: /content/drive/MyDrive/Credit_Risk_Thesis/data/raw/accepted_2007_to_2018Q4.csv


In [8]:
required_columns = (
    candidate_features
    + ["loan_status", "issue_d"]
)

# Remove duplicates while preserving order
required_columns = list(
    dict.fromkeys(required_columns)
)

print("Candidate source features:", len(candidate_features))
print("Additional required columns:", ["loan_status", "issue_d"])
print("Total columns to load:", len(required_columns))

Candidate source features: 64
Additional required columns: ['loan_status', 'issue_d']
Total columns to load: 66


In [9]:
df = pd.read_csv(
    RAW_DATA_FILE,
    usecols=required_columns,
    low_memory=False
)

print("Loaded dataset shape:", df.shape)

Loaded dataset shape: (2260701, 66)


In [10]:
assert df.shape[1] == 66

missing_required_columns = sorted(
    set(required_columns) - set(df.columns)
)

assert len(missing_required_columns) == 0

print("All required columns loaded successfully.")

All required columns loaded successfully.


In [11]:
df.head()

,loan_amnt,term,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,desc,purpose,...,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit
0,3600.0,36 months,10+ years,MORTGAGE,55000.0,Not Verified,Dec-2015,Fully Paid,NaN,debt_consolidation,...,0.0,3.0,76.9,0.0,0.0,0.0,178050.0,7746.0,2400.0,13734.0
1,24700.0,36 months,10+ years,MORTGAGE,65000.0,Not Verified,Dec-2015,Fully Paid,NaN,small_business,...,0.0,2.0,97.4,7.7,0.0,0.0,314017.0,39475.0,79300.0,24667.0
2,20000.0,60 months,10+ years,MORTGAGE,63000.0,Not Verified,Dec-2015,Fully Paid,NaN,home_improvement,...,0.0,0.0,100.0,50.0,0.0,0.0,218418.0,18696.0,6200.0,14877.0
3,35000.0,60 months,10+ years,MORTGAGE,110000.0,Source Verified,Dec-2015,Current,NaN,debt_consolidation,...,0.0,1.0,100.0,0.0,0.0,0.0,381215.0,52226.0,62500.0,18000.0
4,10400.0,60 months,3 years,MORTGAGE,104433.0,Source Verified,Dec-2015,Fully Paid,NaN,major_purchase,...,0.0,4.0,96.6,60.0,0.0,0.0,439570.0,95768.0,20300.0,88097.0


## 3. Construction of the Binary Modelling Population

The original dataset contains loans at different stages of their lifecycle. A binary default-prediction experiment requires an unambiguous outcome; therefore, loan-status categories are examined before constructing the modelling population.

Only loans with sufficiently resolved outcomes will be used to construct the binary target. Loans whose final repayment outcome remains uncertain are excluded rather than being arbitrarily assigned to either class.

In [12]:
loan_status_summary = (
    df["loan_status"]
    .value_counts(dropna=False)
    .rename_axis("loan_status")
    .reset_index(name="count")
)

loan_status_summary["percentage"] = (
    loan_status_summary["count"]
    / len(df)
    * 100
).round(2)

loan_status_summary

,loan_status,count,percentage
0,Fully Paid,1076751,47.63
1,Current,878317,38.85
2,Charged Off,268559,11.88
3,Late (31-120 days),21467,0.95
4,In Grace Period,8436,0.37
5,Late (16-30 days),4349,0.19
6,Does not meet the credit policy. Status:Fully ...,1988,0.09
7,Does not meet the credit policy. Status:Charge...,761,0.03
8,Default,40,0.00
9,NaN,33,0.00


### 3.1 Definition of Resolved Outcomes

For the primary binary classification task, the modelling population is restricted to loans with unambiguous final outcomes under the main lending population:

- `Fully Paid` is defined as the non-default class (`0`).
- `Charged Off` is defined as the default class (`1`).

Active or intermediate statuses (`Current`, late-payment categories, and `In Grace Period`) are excluded because their ultimate repayment outcome is unresolved.

Records associated with the historical "Does not meet the credit policy" categories are excluded to avoid mixing a distinct credit-policy population with the primary modelling sample. The very small number of records labelled `Default` are also excluded because this status represents a distinct loan-state category rather than the finalized `Charged Off` outcome used to define default in the primary experiment.

Records with missing loan status are excluded.

In [13]:
resolved_statuses = [
    "Fully Paid",
    "Charged Off"
]

model_df = (
    df[df["loan_status"].isin(resolved_statuses)]
    .copy()
)

print("Original observations:", len(df))
print("Resolved observations:", len(model_df))
print(
    "Excluded observations:",
    len(df) - len(model_df)
)

Original observations: 2260701
Resolved observations: 1345310
Excluded observations: 915391


In [14]:
model_df["default_flag"] = (
    model_df["loan_status"]
    .map({
        "Fully Paid": 0,
        "Charged Off": 1
    })
    .astype("int8")
)

In [15]:
target_summary = (
    model_df["default_flag"]
    .value_counts()
    .sort_index()
    .rename_axis("default_flag")
    .reset_index(name="count")
)

target_summary["percentage"] = (
    target_summary["count"]
    / len(model_df)
    * 100
).round(2)

target_summary

,default_flag,count,percentage
0,0,1076751,80.04
1,1,268559,19.96


#### Target Distribution Interpretation

After restricting the dataset to loans with resolved outcomes, the modelling population contains 1,345,310 observations.

- 1,076,751 loans (80.04%) were fully paid and are labelled as non-default (`0`).
- 268,559 loans (19.96%) were charged off and are labelled as default (`1`).

The resulting target distribution shows moderate class imbalance, with approximately one default observation for every four non-default observations. No resampling or class-balancing transformation is applied at this stage. Any imbalance-handling strategy will be evaluated later using only the training data to avoid information leakage.

In [16]:
TEXT_FEATURE = "desc"

assert TEXT_FEATURE in candidate_features

structured_features = [
    col for col in candidate_features
    if col != TEXT_FEATURE
]

print("Total candidate features:", len(candidate_features))
print("Structured features:", len(structured_features))
print("Text features:", 1)

print("\nText feature:")
print(TEXT_FEATURE)

Total candidate features: 64
Structured features: 63
Text features: 1

Text feature:
desc


## 4. Feature Engineering

### 4.1 Consolidation of the FICO Score Range

The retained source variables `fico_range_low` and `fico_range_high` represent the lower and upper boundaries of the borrower's FICO score range. Notebook 02 identified these variables as perfectly correlated.

Rather than retaining both highly redundant boundaries, a single representative FICO score is constructed as the midpoint of the reported range:

\[
\text{fico\_score} =
\frac{\text{fico\_range\_low} + \text{fico\_range\_high}}{2}
\]

The original range variables are subsequently excluded from the structured modelling feature set.

In [17]:
model_df["fico_score"] = (
    model_df["fico_range_low"]
    + model_df["fico_range_high"]
) / 2

In [18]:
model_df[
    [
        "fico_range_low",
        "fico_range_high",
        "fico_score"
    ]
].head(10)

,fico_range_low,fico_range_high,fico_score
0,675.0,679.0,677.0
1,715.0,719.0,717.0
2,695.0,699.0,697.0
4,695.0,699.0,697.0
5,690.0,694.0,692.0
6,680.0,684.0,682.0
7,705.0,709.0,707.0
8,685.0,689.0,687.0
9,700.0,704.0,702.0
12,700.0,704.0,702.0


In [19]:
model_df[
    [
        "fico_range_low",
        "fico_range_high",
        "fico_score"
    ]
].describe()

,fico_range_low,fico_range_high,fico_score
count,1.345310e+06,1.345310e+06,1.345310e+06
mean,6.961850e+02,7.001852e+02,6.981851e+02
std,3.185251e+01,3.185316e+01,3.185284e+01
min,6.250000e+02,6.290000e+02,6.270000e+02
25%,6.700000e+02,6.740000e+02,6.720000e+02
50%,6.900000e+02,6.940000e+02,6.920000e+02
75%,7.100000e+02,7.140000e+02,7.120000e+02
max,8.450000e+02,8.500000e+02,8.475000e+02


In [20]:
structured_features = [
    col for col in structured_features
    if col not in [
        "fico_range_low",
        "fico_range_high"
    ]
]

structured_features.append("fico_score")

print("Structured features after FICO consolidation:",
      len(structured_features))

print("FICO low retained:",
      "fico_range_low" in structured_features)

print("FICO high retained:",
      "fico_range_high" in structured_features)

print("Engineered FICO retained:",
      "fico_score" in structured_features)

Structured features after FICO consolidation: 62
FICO low retained: False
FICO high retained: False
Engineered FICO retained: True


### 4.2 Credit-History Length

The raw `earliest_cr_line` variable represents the date on which the borrower's earliest reported credit line was opened. Using the raw calendar date directly would provide limited interpretable information and introduce a temporal representation into the model.

Instead, credit-history length is derived relative to the loan issue date (`issue_d`). The issue date is used solely as an auxiliary reference for this transformation and is not retained as a predictor.

Before constructing the derived feature, both date variables are inspected and converted to appropriate datetime representations.

In [21]:
model_df[
    ["earliest_cr_line", "issue_d"]
].head(10)

,earliest_cr_line,issue_d
0,Aug-2003,Dec-2015
1,Dec-1999,Dec-2015
2,Aug-2000,Dec-2015
4,Jun-1998,Dec-2015
5,Oct-1987,Dec-2015
6,Jun-1990,Dec-2015
7,Feb-1999,Dec-2015
8,Apr-2002,Dec-2015
9,Nov-1994,Dec-2015
12,Jun-1996,Dec-2015


In [22]:
print("earliest_cr_line examples:")
print(
    model_df["earliest_cr_line"]
    .dropna()
    .unique()[:10]
)

print("\nissue_d examples:")
print(
    model_df["issue_d"]
    .dropna()
    .unique()[:10]
)

earliest_cr_line examples:
['Aug-2003' 'Dec-1999' 'Aug-2000' 'Jun-1998' 'Oct-1987' 'Jun-1990'
 'Feb-1999' 'Apr-2002' 'Nov-1994' 'Jun-1996']

issue_d examples:
['Dec-2015' 'Nov-2015' 'Oct-2015' 'Sep-2015' 'Aug-2015' 'Jul-2015'
 'Jun-2015' 'May-2015' 'Apr-2015' 'Mar-2015']


In [23]:
model_df["earliest_cr_line_date"] = pd.to_datetime(
    model_df["earliest_cr_line"],
    format="%b-%Y",
    errors="coerce"
)

model_df["issue_date"] = pd.to_datetime(
    model_df["issue_d"],
    format="%b-%Y",
    errors="coerce"
)

In [24]:
date_validation = pd.DataFrame({
    "variable": ["earliest_cr_line", "issue_d"],
    "original_missing": [
        model_df["earliest_cr_line"].isna().sum(),
        model_df["issue_d"].isna().sum()
    ],
    "parsed_missing": [
        model_df["earliest_cr_line_date"].isna().sum(),
        model_df["issue_date"].isna().sum()
    ],
    "min_date": [
        model_df["earliest_cr_line_date"].min(),
        model_df["issue_date"].min()
    ],
    "max_date": [
        model_df["earliest_cr_line_date"].max(),
        model_df["issue_date"].max()
    ]
})

date_validation

,variable,original_missing,parsed_missing,min_date,max_date
0,earliest_cr_line,0,0,1934-04-01,2015-10-01
1,issue_d,0,0,2007-06-01,2018-12-01


In [25]:
invalid_date_order = (
    model_df["earliest_cr_line_date"]
    > model_df["issue_date"]
)

print(
    "Earliest credit line after loan issue date:",
    invalid_date_order.sum()
)

Earliest credit line after loan issue date: 0


In [26]:
model_df.loc[
    invalid_date_order,
    [
        "earliest_cr_line",
        "issue_d",
        "earliest_cr_line_date",
        "issue_date"
    ]
].head(20)

,earliest_cr_line,issue_d,earliest_cr_line_date,issue_date


In [27]:
model_df["credit_history_months"] = (
    (model_df["issue_date"].dt.year -
     model_df["earliest_cr_line_date"].dt.year) * 12
    +
    (model_df["issue_date"].dt.month -
     model_df["earliest_cr_line_date"].dt.month)
)

In [28]:
model_df["credit_history_months"].describe()

,credit_history_months
count,1.345310e+06
mean,1.951127e+02
std,9.007263e+01
min,3.600000e+01
25%,1.350000e+02
50%,1.770000e+02
75%,2.400000e+02
max,9.990000e+02


In [29]:
print(
    "Missing credit history:",
    model_df["credit_history_months"].isna().sum()
)

print(
    "Negative credit history:",
    (model_df["credit_history_months"] < 0).sum()
)

print(
    "Minimum credit history:",
    model_df["credit_history_months"].min()
)

print(
    "Maximum credit history:",
    model_df["credit_history_months"].max()
)

Missing credit history: 0
Negative credit history: 0
Minimum credit history: 36
Maximum credit history: 999


In [30]:
structured_features = [
    col for col in structured_features
    if col != "earliest_cr_line"
]

structured_features.append("credit_history_months")

print(
    "Structured features after credit-history engineering:",
    len(structured_features)
)

print(
    "earliest_cr_line retained:",
    "earliest_cr_line" in structured_features
)

print(
    "credit_history_months retained:",
    "credit_history_months" in structured_features
)

print(
    "issue_d retained:",
    "issue_d" in structured_features
)

Structured features after credit-history engineering: 62
earliest_cr_line retained: False
credit_history_months retained: True
issue_d retained: False


#### Credit-History Engineering Result

Both `earliest_cr_line` and the auxiliary `issue_d` variable were successfully parsed using an explicit month-year format, with no additional missing values introduced during conversion. No observations were identified in which the earliest credit-line date occurred after the corresponding loan issue date.

A derived variable, `credit_history_months`, was therefore constructed as the number of calendar months between the borrower's earliest reported credit line and the loan issue date. This provides an interpretable measure of credit-history length available at the time of underwriting.

The original `earliest_cr_line` variable is replaced by `credit_history_months` in the structured modelling feature set. `issue_d` is used only as an auxiliary variable for this derivation and is not retained as a model predictor.

## 5. Structured Feature Classification

Following feature engineering, the retained structured predictors are classified according to their data types. This separation is required because numerical and categorical variables require different preprocessing strategies.

Feature classification is performed programmatically from the engineered modelling dataset to reduce the risk of manually omitting or misclassifying predictors.

In [31]:
numerical_features = (
    model_df[structured_features]
    .select_dtypes(include="number")
    .columns
    .tolist()
)

categorical_features = (
    model_df[structured_features]
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

print("Total structured features:", len(structured_features))
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

Total structured features: 62
Numerical features: 56
Categorical features: 6


In [32]:
classified_features = (
    numerical_features
    + categorical_features
)

unclassified_features = sorted(
    set(structured_features)
    - set(classified_features)
)

duplicate_classification = (
    set(numerical_features)
    & set(categorical_features)
)

print("Unclassified features:", unclassified_features)
print(
    "Features appearing in both groups:",
    duplicate_classification
)

assert len(classified_features) == len(structured_features)
assert len(unclassified_features) == 0
assert len(duplicate_classification) == 0

print("\nStructured feature classification validated.")

Unclassified features: []
Features appearing in both groups: set()

Structured feature classification validated.


In [33]:
categorical_summary = pd.DataFrame({
    "feature": categorical_features,
    "dtype": [
        str(model_df[col].dtype)
        for col in categorical_features
    ],
    "missing_count": [
        model_df[col].isna().sum()
        for col in categorical_features
    ],
    "missing_pct": [
        round(model_df[col].isna().mean() * 100, 2)
        for col in categorical_features
    ],
    "unique_values": [
        model_df[col].nunique(dropna=True)
        for col in categorical_features
    ]
})

categorical_summary

,feature,dtype,missing_count,missing_pct,unique_values
0,term,object,0,0.00,2
1,emp_length,object,78511,5.84,11
2,home_ownership,object,0,0.00,6
3,verification_status,object,0,0.00,3
4,purpose,object,0,0.00,14
5,application_type,object,0,0.00,2


In [34]:
model_df["emp_length"].value_counts(
    dropna=False
)

,count
emp_length,
10+ years,442199
2 years,121743
< 1 year,108061
3 years,107597
1 year,88494
5 years,84154
4 years,80556
NaN,78511
6 years,62733


In [35]:
emp_length_mapping = {
    "< 1 year": 0,
    "1 year": 1,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "8 years": 8,
    "9 years": 9,
    "10+ years": 10
}

model_df["emp_length_years"] = (
    model_df["emp_length"]
    .map(emp_length_mapping)
)

In [36]:
emp_length_validation = pd.DataFrame({
    "original_missing": [
        model_df["emp_length"].isna().sum()
    ],
    "engineered_missing": [
        model_df["emp_length_years"].isna().sum()
    ],
    "engineered_min": [
        model_df["emp_length_years"].min()
    ],
    "engineered_max": [
        model_df["emp_length_years"].max()
    ]
})

emp_length_validation

,original_missing,engineered_missing,engineered_min,engineered_max
0,78511,78511,0.0,10.0


In [37]:
assert (
    model_df["emp_length"].isna().sum()
    ==
    model_df["emp_length_years"].isna().sum()
)

assert model_df["emp_length_years"].min() == 0
assert model_df["emp_length_years"].max() == 10

print("Employment-length transformation validated.")

Employment-length transformation validated.


In [38]:
structured_features = [
    col for col in structured_features
    if col != "emp_length"
]

structured_features.append("emp_length_years")

In [39]:
numerical_features = (
    model_df[structured_features]
    .select_dtypes(include="number")
    .columns
    .tolist()
)

categorical_features = (
    model_df[structured_features]
    .select_dtypes(include=["object", "category"])
    .columns
    .tolist()
)

print("Structured features:", len(structured_features))
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

print("\nCategorical features:")
print(categorical_features)

Structured features: 62
Numerical features: 57
Categorical features: 5

Categorical features:
['term', 'home_ownership', 'verification_status', 'purpose', 'application_type']


#### Employment-Length Engineering Result

`emp_length` contained 11 ordered non-missing categories ranging from less than one year to ten or more years. The variable was therefore transformed into `emp_length_years`, with values from 0 to 10, preserving the ordinal nature of employment tenure.

The transformation introduced no additional missing values. Missing employment-tenure observations are intentionally retained at this stage and will be imputed later using parameters estimated from the training data only.

Following this transformation, the structured feature set contains 62 predictors: 57 numerical and 5 nominal categorical variables.

## 6. Missing-Value Assessment and Preprocessing Strategy

Missingness is reassessed after feature engineering because the final structured modelling variables differ from the original source feature set.

At this stage, missing values are characterized but not imputed. Imputation parameters will subsequently be estimated using the training partition only and then applied unchanged to the validation and test partitions. This prevents information from the evaluation data from influencing preprocessing decisions.

In [40]:
structured_missingness = pd.DataFrame({
    "feature": structured_features,
    "dtype": [
        str(model_df[col].dtype)
        for col in structured_features
    ],
    "missing_count": [
        model_df[col].isna().sum()
        for col in structured_features
    ],
    "missing_pct": [
        round(model_df[col].isna().mean() * 100, 2)
        for col in structured_features
    ]
}).sort_values(
    "missing_pct",
    ascending=False
).reset_index(drop=True)

structured_missingness

,feature,dtype,missing_count,missing_pct
0,mths_since_last_record,float64,1116755,83.01
1,mths_since_recent_bc_dlq,float64,1026290,76.29
2,mths_since_last_major_derog,float64,991560,73.70
3,mths_since_recent_revol_delinq,float64,895348,66.55
4,mths_since_last_delinq,float64,678743,50.45
...,...,...,...,...
57,delinq_2yrs,float64,0,0.00
58,inq_last_6mths,float64,1,0.00
59,tax_liens,float64,39,0.00
60,fico_score,float64,0,0.00


In [41]:
missingness_summary = pd.Series({
    "0% missing":
        (structured_missingness["missing_pct"] == 0).sum(),

    ">0% to 5%":
        (
            (structured_missingness["missing_pct"] > 0) &
            (structured_missingness["missing_pct"] <= 5)
        ).sum(),

    ">5% to 10%":
        (
            (structured_missingness["missing_pct"] > 5) &
            (structured_missingness["missing_pct"] <= 10)
        ).sum(),

    ">10% to 30%":
        (
            (structured_missingness["missing_pct"] > 10) &
            (structured_missingness["missing_pct"] <= 30)
        ).sum(),

    ">30%":
        (structured_missingness["missing_pct"] > 30).sum()
})

missingness_summary

,0
0% missing,20
>0% to 5%,12
>5% to 10%,24
>10% to 30%,1
>30%,5


In [42]:
print(
    "Maximum structured-feature missingness:",
    structured_missingness["missing_pct"].max(),
    "%"
)

structured_missingness[
    structured_missingness["missing_pct"] > 0
]

Maximum structured-feature missingness: 83.01 %


,feature,dtype,missing_count,missing_pct
0,mths_since_last_record,float64,1116755,83.01
1,mths_since_recent_bc_dlq,float64,1026290,76.29
2,mths_since_last_major_derog,float64,991560,73.70
3,mths_since_recent_revol_delinq,float64,895348,66.55
4,mths_since_last_delinq,float64,678743,50.45
5,mths_since_recent_inq,float64,174071,12.94
6,num_tl_120dpd_2m,float64,117401,8.73
7,mo_sin_old_il_acct,float64,105575,7.85
8,emp_length_years,float64,78511,5.84
9,pct_tl_nvr_dlq,float64,67681,5.03


### 6.1 Missing-Value Treatment Strategy

The retained structured variables exhibit heterogeneous missingness patterns. In particular, several credit-history recency variables contain substantial missingness because the underlying event may not have occurred for a borrower. Consequently, absence in these variables can itself carry predictive information.

Several event-recency variables exhibit substantial or potentially informative missingness. Exploratory analysis of mths_since_recent_inq, for example, showed that missing observations were strongly associated with zero recent enquiries, although the relationship was not deterministic. Accordingly, event-recency variables are retained and their missingness is explicitly represented through indicator features rather than interpreted as direct evidence that an event never occurred.

For other numerical variables, missing values will be imputed using statistics estimated exclusively from the training partition. Categorical missingness will similarly be handled within the training-fitted preprocessing pipeline.

In [43]:
history_recency_features = [
    "mths_since_last_record",
    "mths_since_recent_bc_dlq",
    "mths_since_last_major_derog",
    "mths_since_recent_revol_delinq",
    "mths_since_last_delinq"
]

assert all(
    col in numerical_features
    for col in history_recency_features
)

print(
    "History-recency features:",
    len(history_recency_features)
)

History-recency features: 5


In [44]:
model_df[
    [
        "mths_since_recent_inq",
        "inq_last_6mths"
    ]
].head(20)

,mths_since_recent_inq,inq_last_6mths
0,4.0,1.0
1,0.0,4.0
2,10.0,0.0
4,1.0,3.0
5,NaN,0.0
6,10.0,0.0
7,8.0,0.0
8,1.0,1.0
9,10.0,0.0
12,18.0,0.0


In [45]:
pd.crosstab(
    model_df["mths_since_recent_inq"].isna(),
    model_df["inq_last_6mths"],
    normalize="index"
).mul(100).round(2)

inq_last_6mths,0.0,1.0,2.0,3.0,4.0,5.0,6.0,7.0,8.0
mths_since_recent_inq,,,,,,,,,
False,52.99,30.18,11.10,4.11,1.15,0.4,0.07,0.00,0.00
True,86.05,7.61,3.93,2.04,0.21,0.1,0.04,0.02,0.01


In [46]:
model_df.groupby(
    model_df["mths_since_recent_inq"].isna()
)["inq_last_6mths"].agg(
    ["count", "mean", "median", "min", "max"]
)

,count,mean,median,min,max
mths_since_recent_inq,,,,,
False,1171238,0.717706,0.0,0.0,8.0
True,174071,0.233703,0.0,0.0,8.0


In [47]:
history_recency_features = [
    "mths_since_last_record",
    "mths_since_recent_bc_dlq",
    "mths_since_last_major_derog",
    "mths_since_recent_revol_delinq",
    "mths_since_last_delinq",
    "mths_since_recent_inq"
]

assert all(
    col in numerical_features
    for col in history_recency_features
)

print(
    "History/event-recency features requiring "
    "missingness indicators:",
    len(history_recency_features)
)

History/event-recency features requiring missingness indicators: 6


In [48]:
# Create issue year for temporal analysis
model_df["issue_year"] = model_df["issue_date"].dt.year

temporal_summary = (
    model_df
    .groupby("issue_year")
    .agg(
        loan_count=("default_flag", "size"),
        default_count=("default_flag", "sum"),
        default_rate=("default_flag", "mean")
    )
    .reset_index()
)

temporal_summary["default_rate_pct"] = (
    temporal_summary["default_rate"] * 100
).round(2)

temporal_summary[
    [
        "issue_year",
        "loan_count",
        "default_count",
        "default_rate_pct"
    ]
]

,issue_year,loan_count,default_count,default_rate_pct
0,2007,251,45,17.93
1,2008,1562,247,15.81
2,2009,4716,594,12.60
3,2010,11536,1487,12.89
4,2011,21721,3297,15.18
5,2012,53367,8644,16.20
6,2013,134804,21024,15.60
7,2014,223102,41161,18.45
8,2015,375545,75803,20.18
9,2016,293095,68242,23.28


## 7. Temporal Split and Out-of-Time Evaluation

### 7.1 Temporal Split Decision

The resolved-loan population exhibits a noticeable temporal change in default prevalence. Default rates generally increased in the later years, reaching approximately 23% in 2016–2017, while the 2018 cohort exhibited a lower default rate of approximately 15.8%.

To approximate a realistic credit-risk deployment setting and avoid information from future loan cohorts influencing model development, a chronological out-of-time split is adopted:

- **Training set:** loans issued from 2007 to 2016
- **Validation set:** loans issued in 2017
- **Test set:** loans issued in 2018

The validation set is used for model selection and hyperparameter-related decisions, while the 2018 cohort is reserved as an unseen out-of-time test set. The change in default prevalence across these periods also provides a useful test of model robustness under temporal distribution shift.


In [49]:
train_mask = model_df["issue_year"] <= 2016
val_mask = model_df["issue_year"] == 2017
test_mask = model_df["issue_year"] == 2018

train_df = model_df.loc[train_mask].copy()
val_df = model_df.loc[val_mask].copy()
test_df = model_df.loc[test_mask].copy()

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

Train shape: (1119699, 73)
Validation shape: (169300, 73)
Test shape: (56311, 73)


In [50]:
split_summary = pd.DataFrame({
    "split": ["Train", "Validation", "Test"],
    "rows": [
        len(train_df),
        len(val_df),
        len(test_df)
    ],
    "default_count": [
        train_df["default_flag"].sum(),
        val_df["default_flag"].sum(),
        test_df["default_flag"].sum()
    ],
    "default_rate_pct": [
        train_df["default_flag"].mean() * 100,
        val_df["default_flag"].mean() * 100,
        test_df["default_flag"].mean() * 100
    ],
    "min_year": [
        train_df["issue_year"].min(),
        val_df["issue_year"].min(),
        test_df["issue_year"].min()
    ],
    "max_year": [
        train_df["issue_year"].max(),
        val_df["issue_year"].max(),
        test_df["issue_year"].max()
    ]
})

split_summary["default_rate_pct"] = (
    split_summary["default_rate_pct"].round(2)
)

split_summary

,split,rows,default_count,default_rate_pct,min_year,max_year
0,Train,1119699,220544,19.70,2007,2016
1,Validation,169300,39148,23.12,2017,2017
2,Test,56311,8867,15.75,2018,2018


In [51]:
assert len(train_df) + len(val_df) + len(test_df) == len(model_df)
assert train_df["issue_year"].max() < val_df["issue_year"].min()
assert val_df["issue_year"].max() < test_df["issue_year"].min()
assert set(train_df.index).isdisjoint(val_df.index)
assert set(train_df.index).isdisjoint(test_df.index)
assert set(val_df.index).isdisjoint(test_df.index)

print("Temporal split validation passed.")


Temporal split validation passed.


## 8. Train-Only Structured Preprocessing

All preprocessing operations that estimate parameters from the data are fitted using the training partition only. The fitted transformations are subsequently applied unchanged to the validation and out-of-time test partitions. This prevents information leakage from future loan cohorts into model development.

The target variable and textual modality are separated from the structured predictor set before fitting any preprocessing transformations.


In [52]:
X_train = train_df[structured_features].copy()
X_val = val_df[structured_features].copy()
X_test = test_df[structured_features].copy()

y_train = train_df["default_flag"].copy()
y_val = val_df["default_flag"].copy()
y_test = test_df["default_flag"].copy()

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print()
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (1119699, 62)
X_val: (169300, 62)
X_test: (56311, 62)

y_train: (1119699,)
y_val: (169300,)
y_test: (56311,)


In [53]:
assert X_train.shape[1] == 62
assert X_val.shape[1] == 62
assert X_test.shape[1] == 62
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)
assert "default_flag" not in X_train.columns
assert "desc" not in X_train.columns
assert "issue_d" not in X_train.columns
assert "issue_year" not in X_train.columns
assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns)

print("Structured train/validation/test matrices validated.")


Structured train/validation/test matrices validated.


In [54]:
text_train = train_df["desc"].copy()
text_val = val_df["desc"].copy()
text_test = test_df["desc"].copy()

print("Training text rows:", len(text_train))
print("Validation text rows:", len(text_val))
print("Test text rows:", len(text_test))

Training text rows: 1119699
Validation text rows: 169300
Test text rows: 56311


### 8.1 Missing-Value Treatment Strategy

Missing-value treatment is determined using the training partition only to prevent information leakage from the validation and out-of-time test cohorts.

Structured numerical predictors are divided into two groups:

1. **Ordinary numerical features** — missing values are imputed using the median estimated from the training data.
2. **Event-recency features** — variables describing the number of months since a historical credit event. Their missingness may itself contain information about the borrower's observed credit history. For these variables, a binary missingness indicator is created before median imputation.

The five retained categorical predictors contain no missing values in the temporal partitions, so categorical imputation is not required. All learned preprocessing parameters are estimated exclusively from the training partition and subsequently applied unchanged to validation and test data.


In [55]:
history_recency_features = [
    "mths_since_last_record",
    "mths_since_recent_bc_dlq",
    "mths_since_last_major_derog",
    "mths_since_recent_revol_delinq",
    "mths_since_last_delinq",
    "mths_since_recent_inq"
]

In [56]:
ordinary_numerical_features = [
    col for col in numerical_features
    if col not in history_recency_features
]

print("Total numerical features:", len(numerical_features))
print(
    "Event-recency numerical features:",
    len(history_recency_features)
)
print(
    "Ordinary numerical features:",
    len(ordinary_numerical_features)
)

assert set(history_recency_features).issubset(
    set(numerical_features)
)

assert set(ordinary_numerical_features).isdisjoint(
    set(history_recency_features)
)

assert (
    len(ordinary_numerical_features)
    + len(history_recency_features)
    == len(numerical_features)
)

print("\nNumerical feature grouping validated.")

Total numerical features: 57
Event-recency numerical features: 6
Ordinary numerical features: 51

Numerical feature grouping validated.


In [57]:
X_train_processed = X_train.copy()
X_val_processed = X_val.copy()
X_test_processed = X_test.copy()

missing_indicator_features = []

for col in history_recency_features:
    indicator_col = f"{col}_missing"

    X_train_processed[indicator_col] = (
        X_train_processed[col].isna().astype("int8")
    )

    X_val_processed[indicator_col] = (
        X_val_processed[col].isna().astype("int8")
    )

    X_test_processed[indicator_col] = (
        X_test_processed[col].isna().astype("int8")
    )

    missing_indicator_features.append(indicator_col)

print(
    "Missingness indicators created:",
    len(missing_indicator_features)
)

print(missing_indicator_features)

Missingness indicators created: 6
['mths_since_last_record_missing', 'mths_since_recent_bc_dlq_missing', 'mths_since_last_major_derog_missing', 'mths_since_recent_revol_delinq_missing', 'mths_since_last_delinq_missing', 'mths_since_recent_inq_missing']


In [58]:
assert len(missing_indicator_features) == 6

assert X_train_processed.shape[1] == 68
assert X_val_processed.shape[1] == 68
assert X_test_processed.shape[1] == 68

for col in missing_indicator_features:
    assert set(
        X_train_processed[col].unique()
    ).issubset({0, 1})

print("Missingness indicators validated.")

Missingness indicators validated.


### 8.2 Training-Only Numerical Imputation

Numerical imputation parameters are estimated exclusively from the training
partition. Median imputation is used because it is less sensitive to extreme
values than mean imputation, which is appropriate for several skewed financial
and credit-history variables in the dataset.

For event-recency variables, the previously created missingness indicators
preserve information about whether the original value was observed. The
numerical value itself is subsequently median-imputed.

The same training-derived median values are applied unchanged to the validation
and out-of-time test partitions.


In [59]:
# Learn numerical imputation values from TRAINING DATA ONLY

train_numerical_medians = (
    X_train_processed[numerical_features]
    .median()
)

print(
    "Numerical imputation values learned:",
    len(train_numerical_medians)
)

train_numerical_medians.head(10)

Numerical imputation values learned: 57


,0
loan_amnt,12000.00
annual_inc,65000.00
dti,17.65
delinq_2yrs,0.00
inq_last_6mths,0.00
mths_since_last_delinq,31.00
mths_since_last_record,70.00
open_acc,11.00
pub_rec,0.00
revol_bal,11369.00


In [60]:
missing_medians = (
    train_numerical_medians[
        train_numerical_medians.isna()
    ]
)

print(
    "Features without a valid training median:",
    len(missing_medians)
)

if len(missing_medians) > 0:
    print(missing_medians)

Features without a valid training median: 0


In [61]:
X_train_processed[numerical_features] = (
    X_train_processed[numerical_features]
    .fillna(train_numerical_medians)
)

X_val_processed[numerical_features] = (
    X_val_processed[numerical_features]
    .fillna(train_numerical_medians)
)

X_test_processed[numerical_features] = (
    X_test_processed[numerical_features]
    .fillna(train_numerical_medians)
)

In [62]:
numerical_missing_after = pd.DataFrame({
    "train": X_train_processed[
        numerical_features
    ].isna().sum(),

    "validation": X_val_processed[
        numerical_features
    ].isna().sum(),

    "test": X_test_processed[
        numerical_features
    ].isna().sum()
})

print(
    "Remaining numerical missing values:"
)

print(numerical_missing_after.sum())

Remaining numerical missing values:
train         0
validation    0
test          0
dtype: int64


In [63]:
assert X_train_processed[numerical_features].isna().sum().sum() == 0
assert X_val_processed[numerical_features].isna().sum().sum() == 0
assert X_test_processed[numerical_features].isna().sum().sum() == 0

print("Numerical imputation validated.")


Numerical imputation validated.


In [64]:
indicator_summary = pd.DataFrame({
    "feature": history_recency_features,
    "indicator": missing_indicator_features,
    "train_missing_indicator_count": [
        X_train_processed[ind].sum()
        for ind in missing_indicator_features
    ]
})

indicator_summary

,feature,indicator,train_missing_indicator_count
0,mths_since_last_record,mths_since_last_record_missing,932398
1,mths_since_recent_bc_dlq,mths_since_recent_bc_dlq_missing,851835
2,mths_since_last_major_derog,mths_since_last_major_derog_missing,826426
3,mths_since_recent_revol_delinq,mths_since_recent_revol_delinq_missing,743502
4,mths_since_last_delinq,mths_since_last_delinq_missing,563251
5,mths_since_recent_inq,mths_since_recent_inq_missing,153979


### 8.3 Categorical Feature Preparation

The original `emp_length` source variable was transformed into the numerical `emp_length_years` representation during feature engineering and therefore no longer appears in the categorical feature list.

The remaining five categorical variables are retained for encoding. Diagnostic checks confirm that these variables contain no missing values in the training, validation, or test partitions. Category availability is also compared across temporal partitions before fitting the encoder.


In [65]:
print("Categorical features before adjustment:")
print(categorical_features)

categorical_features_final = categorical_features.copy()

print("Final categorical features:")
print(categorical_features_final)

print(
    "\nCategorical feature count:",
    len(categorical_features_final)
)

Categorical features before adjustment:
['term', 'home_ownership', 'verification_status', 'purpose', 'application_type']
Final categorical features:
['term', 'home_ownership', 'verification_status', 'purpose', 'application_type']

Categorical feature count: 5


In [66]:
categorical_missingness = pd.DataFrame({
    "feature": categorical_features_final,

    "train_missing": [
        X_train_processed[col].isna().sum()
        for col in categorical_features_final
    ],

    "validation_missing": [
        X_val_processed[col].isna().sum()
        for col in categorical_features_final
    ],

    "test_missing": [
        X_test_processed[col].isna().sum()
        for col in categorical_features_final
    ]
})

categorical_missingness["train_missing_pct"] = (
    categorical_missingness["train_missing"]
    / len(X_train_processed) * 100
).round(2)

categorical_missingness

,feature,train_missing,validation_missing,test_missing,train_missing_pct
0,term,0,0,0,0.0
1,home_ownership,0,0,0,0.0
2,verification_status,0,0,0,0.0
3,purpose,0,0,0,0.0
4,application_type,0,0,0,0.0


In [67]:
unseen_category_summary = []

for col in categorical_features_final:

    train_categories = set(
        X_train_processed[col].dropna().unique()
    )

    val_categories = set(
        X_val_processed[col].dropna().unique()
    )

    test_categories = set(
        X_test_processed[col].dropna().unique()
    )

    unseen_category_summary.append({
        "feature": col,
        "train_unique": len(train_categories),
        "validation_unique": len(val_categories),
        "test_unique": len(test_categories),
        "unseen_in_validation": sorted(
            val_categories - train_categories
        ),
        "unseen_in_test": sorted(
            test_categories - train_categories
        )
    })

unseen_category_summary = pd.DataFrame(
    unseen_category_summary
)

unseen_category_summary

,feature,train_unique,validation_unique,test_unique,unseen_in_validation,unseen_in_test
0,term,2,2,2,[],[]
1,home_ownership,6,5,4,[],[]
2,verification_status,3,3,3,[],[]
3,purpose,14,12,13,[],[]
4,application_type,2,2,2,[],[]


In [68]:
unseen_category_summary[
    [
        "feature",
        "unseen_in_validation",
        "unseen_in_test"
    ]
]

,feature,unseen_in_validation,unseen_in_test
0,term,[],[]
1,home_ownership,[],[]
2,verification_status,[],[]
3,purpose,[],[]
4,application_type,[],[]


### 8.4 Categorical Encoding

The five retained categorical predictors contain no missing values and no categories appearing exclusively in the validation or test periods.

One-hot encoding is therefore applied to the categorical variables. The encoder is fitted exclusively on the training partition and then applied unchanged to the validation and out-of-time test partitions.

`handle_unknown="ignore"` is retained as a defensive setting so that future observations containing previously unseen categories can be transformed without refitting the encoder or causing an error.


In [69]:
from sklearn.preprocessing import OneHotEncoder

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

In [70]:
categorical_encoder.fit(
    X_train_processed[categorical_features_final]
)

OneHotEncoder(handle_unknown='ignore')

In [71]:
encoded_categorical_names = (
    categorical_encoder
    .get_feature_names_out(categorical_features_final)
)

print(
    "Original categorical features:",
    len(categorical_features_final)
)

print(
    "Encoded categorical features:",
    len(encoded_categorical_names)
)

print("\nEncoded feature names:")
print(encoded_categorical_names)

Original categorical features: 5
Encoded categorical features: 27

Encoded feature names:
['term_ 36 months' 'term_ 60 months' 'home_ownership_ANY'
 'home_ownership_MORTGAGE' 'home_ownership_NONE' 'home_ownership_OTHER'
 'home_ownership_OWN' 'home_ownership_RENT'
 'verification_status_Not Verified' 'verification_status_Source Verified'
 'verification_status_Verified' 'purpose_car' 'purpose_credit_card'
 'purpose_debt_consolidation' 'purpose_educational'
 'purpose_home_improvement' 'purpose_house' 'purpose_major_purchase'
 'purpose_medical' 'purpose_moving' 'purpose_other'
 'purpose_renewable_energy' 'purpose_small_business' 'purpose_vacation'
 'purpose_wedding' 'application_type_Individual'
 'application_type_Joint App']


In [72]:
X_train_cat = categorical_encoder.transform(
    X_train_processed[categorical_features_final]
)

X_validation_cat = categorical_encoder.transform(
    X_val_processed[categorical_features_final]
)

X_test_cat = categorical_encoder.transform(
    X_test_processed[categorical_features_final]
)

print("Train categorical matrix:", X_train_cat.shape)
print("Validation categorical matrix:", X_validation_cat.shape)
print("Test categorical matrix:", X_test_cat.shape)

Train categorical matrix: (1119699, 27)
Validation categorical matrix: (169300, 27)
Test categorical matrix: (56311, 27)


In [73]:
assert X_train_cat.shape[1] == len(encoded_categorical_names)
assert X_validation_cat.shape[1] == len(encoded_categorical_names)
assert X_test_cat.shape[1] == len(encoded_categorical_names)
assert X_train_cat.shape[1] == X_validation_cat.shape[1] == X_test_cat.shape[1]

print("Categorical encoding validated across all temporal splits.")


Categorical encoding validated across all temporal splits.


## 9. Final Structured Feature Matrix

The final structured representation combines:

1. the 57 retained numerical predictors after feature engineering and training-only median imputation;
2. six missingness-indicator variables for selected semantically meaningful missing-value features; and
3. 27 one-hot encoded columns derived from the five retained categorical predictors.

Categorical encoding was fitted exclusively on the training partition and subsequently applied unchanged to the validation and out-of-time test partitions. The components are combined using sparse matrices to reduce memory requirements.

This produces a common structured feature space for all temporal partitions.


In [74]:
print("Numerical feature count:", len(numerical_features))

print(
    "Train numerical shape:",
    X_train_processed[numerical_features].shape
)

print(
    "Validation numerical shape:",
    X_val_processed[numerical_features].shape
)

print(
    "Test numerical shape:",
    X_test_processed[numerical_features].shape
)

print("\nMissing-indicator features:")
print(missing_indicator_features)

print(
    "Missing-indicator count:",
    len(missing_indicator_features)
)

Numerical feature count: 57
Train numerical shape: (1119699, 57)
Validation numerical shape: (169300, 57)
Test numerical shape: (56311, 57)

Missing-indicator features:
['mths_since_last_record_missing', 'mths_since_recent_bc_dlq_missing', 'mths_since_last_major_derog_missing', 'mths_since_recent_revol_delinq_missing', 'mths_since_last_delinq_missing', 'mths_since_recent_inq_missing']
Missing-indicator count: 6


### 9.1 Assembly of the Final Structured Matrices

The final structured representation combines the imputed numerical predictors,
the six missingness-indicator variables, and the one-hot encoded categorical
features.

Because the encoded categorical matrix is sparse and the dataset contains more
than one million training observations, the final matrices are constructed in
sparse format to reduce memory consumption.

The same feature ordering is maintained across the training, validation, and
out-of-time test partitions.


In [75]:
from scipy import sparse

In [76]:
X_train_num = sparse.csr_matrix(
    X_train_processed[numerical_features].to_numpy()
)

X_val_num = sparse.csr_matrix(
    X_val_processed[numerical_features].to_numpy()
)

X_test_num = sparse.csr_matrix(
    X_test_processed[numerical_features].to_numpy()
)

In [77]:
X_train_missing = sparse.csr_matrix(
    X_train_processed[missing_indicator_features].to_numpy()
)

X_val_missing = sparse.csr_matrix(
    X_val_processed[missing_indicator_features].to_numpy()
)

X_test_missing = sparse.csr_matrix(
    X_test_processed[missing_indicator_features].to_numpy()
)

In [78]:
X_train_final = sparse.hstack(
    [
        X_train_num,
        X_train_missing,
        X_train_cat
    ],
    format="csr"
)

X_val_final = sparse.hstack(
    [
        X_val_num,
        X_val_missing,
        X_validation_cat
    ],
    format="csr"
)

X_test_final = sparse.hstack(
    [
        X_test_num,
        X_test_missing,
        X_test_cat
    ],
    format="csr"
)

In [79]:
print("Final train matrix:", X_train_final.shape)
print("Final validation matrix:", X_val_final.shape)
print("Final test matrix:", X_test_final.shape)

Final train matrix: (1119699, 90)
Final validation matrix: (169300, 90)
Final test matrix: (56311, 90)


In [80]:
assert X_train_final.shape == (1119699, 90)
assert X_val_final.shape == (169300, 90)
assert X_test_final.shape == (56311, 90)
assert X_train_final.shape[1] == X_val_final.shape[1] == X_test_final.shape[1]

print("Final structured matrices validated.")


Final structured matrices validated.


In [81]:
final_structured_feature_names = (
    numerical_features
    + missing_indicator_features
    + encoded_categorical_names.tolist()
)

print(
    "Final structured feature names:",
    len(final_structured_feature_names)
)

Final structured feature names: 90


In [82]:
assert len(final_structured_feature_names) == X_train_final.shape[1]
assert len(set(final_structured_feature_names)) == len(final_structured_feature_names)

print("Final structured feature-name mapping validated.")


Final structured feature-name mapping validated.


### 9.2 Structured Preprocessing Result

The final structured representation contains 90 encoded and imputed features:

- 57 numerical predictors after feature engineering and training-only median imputation;
- 6 explicit missingness indicators for selected event-recency variables; and
- 27 one-hot encoded categorical features.

The same feature ordering and training-fitted transformations are applied to the training, validation, and out-of-time test partitions, producing matrices of dimensions 1,119,699 × 90, 169,300 × 90, and 56,311 × 90 respectively.

The matrices are intentionally **not globally standardized in this notebook**. Scaling is model-dependent and will be fitted on the training partition within model-specific pipelines (for example, for regularized logistic regression) so that tree-based models can use the unscaled representation without unnecessary transformation.

The final ordered feature-name list is retained to support subsequent coefficient analysis, feature-importance reporting, ablation studies, and explainability experiments.


## 10. Text Availability and Comparable Experiment Cohort

The borrower-written `desc` field is the textual modality used in the text-only and hybrid experiments. Because textual descriptions are unavailable for a large proportion of loans, text availability is evaluated explicitly before defining a comparison cohort.

Direct comparisons among structured-only, text-only, and hybrid models must use the **same observations**. Otherwise, performance differences could reflect differences in borrower populations rather than differences in representation.


### 10.1 Text Availability Under the Full Structured Temporal Split

The full structured experiment uses 2007–2016 for training, 2017 for validation, and 2018 for out-of-time testing. Text availability is first examined within these same partitions to determine whether this split can also support text-based evaluation.


In [83]:
def usable_text_mask(series):
    return (
        series.notna()
        & series.astype(str).str.strip().ne("")
    )

text_train_mask_full = usable_text_mask(text_train)
text_val_mask_full = usable_text_mask(text_val)
text_test_mask_full = usable_text_mask(text_test)

text_availability_summary = pd.DataFrame({
    "split": ["Train", "Validation", "Test"],
    "total_rows": [len(text_train), len(text_val), len(text_test)],
    "usable_text_rows": [
        text_train_mask_full.sum(),
        text_val_mask_full.sum(),
        text_test_mask_full.sum()
    ]
})

text_availability_summary["usable_text_pct"] = (
    text_availability_summary["usable_text_rows"]
    / text_availability_summary["total_rows"] * 100
).round(2)

text_availability_summary


,split,total_rows,usable_text_rows,usable_text_pct
0,Train,1119699,123293,11.01
1,Validation,169300,0,0.00
2,Test,56311,0,0.00


The full structured temporal split cannot support text-only or hybrid evaluation because usable borrower descriptions are absent from the 2017 and 2018 partitions. The temporal availability of `desc` is therefore examined by issue year before defining a separate matched text cohort.


In [84]:
text_availability_by_year = (
    model_df.assign(
        usable_desc=(
            model_df["desc"].notna()
            & model_df["desc"].astype(str).str.strip().ne("")
        )
    )
    .groupby("issue_year")
    .agg(
        total_rows=("usable_desc", "size"),
        usable_text_rows=("usable_desc", "sum")
    )
)

text_availability_by_year["usable_text_pct"] = (
    text_availability_by_year["usable_text_rows"]
    / text_availability_by_year["total_rows"]
    * 100
).round(2)

text_availability_by_year


,total_rows,usable_text_rows,usable_text_pct
issue_year,,,
2007,251,246,98.01
2008,1562,1562,100.00
2009,4716,4482,95.04
2010,11536,7594,65.83
2011,21721,12723,58.57
2012,53367,32743,61.35
2013,134804,48726,36.15
2014,223102,15175,6.80
2015,375545,30,0.01


In [85]:
text_cohort_by_year = (
    model_df.assign(
        usable_desc=(
            model_df["desc"].notna()
            & model_df["desc"].astype(str).str.strip().ne("")
        )
    )
    .query("usable_desc")
    .groupby("issue_year")
    .agg(
        rows=("default_flag", "size"),
        defaults=("default_flag", "sum"),
        default_rate=("default_flag", "mean")
    )
)

text_cohort_by_year["default_rate_pct"] = (
    text_cohort_by_year["default_rate"] * 100
).round(2)

text_cohort_by_year[
    ["rows", "defaults", "default_rate_pct"]
]


,rows,defaults,default_rate_pct
issue_year,,,
2007,246,45,18.29
2008,1562,247,15.81
2009,4482,568,12.67
2010,7594,1008,13.27
2011,12723,1957,15.38
2012,32743,5282,16.13
2013,48726,7388,15.16
2014,15175,2405,15.85
2015,30,2,6.67


### 10.2 Matched Text-Cohort Temporal Split

Borrower-description availability declines sharply over time. Usable descriptions are common in the earliest vintages, fall to 6.8% in 2014, become effectively unavailable from 2015 onward, and are absent in 2017–2018.

Consequently, Notebook 03 defines two distinct temporal evaluation cohorts:

- **Full structured cohort:** training = 2007–2016, validation = 2017, test = 2018.
- **Matched text-eligible cohort:** training = 2007–2012, validation = 2013, test = 2014, restricted to observations containing a usable `desc`.

The matched text cohort is used for direct structured-only, text-only, and hybrid comparisons so that all representations are evaluated on the same borrowers. The text-based findings therefore apply specifically to loans for which borrower-written descriptions are available and should not be interpreted as representative of the complete Lending Club population.


In [86]:
# Define usable borrower-description mask on the resolved modelling population
usable_desc_mask = (
    model_df["desc"].notna()
    & model_df["desc"].astype(str).str.strip().ne("")
)

# Matched text-eligible temporal cohort
text_train_mask = (
    usable_desc_mask
    & model_df["issue_year"].between(2007, 2012)
)

text_val_mask = (
    usable_desc_mask
    & (model_df["issue_year"] == 2013)
)

text_test_mask = (
    usable_desc_mask
    & (model_df["issue_year"] == 2014)
)

# Preserve original model_df indices for downstream notebook reconstruction
text_train_indices = model_df.index[text_train_mask].to_numpy()
text_val_indices = model_df.index[text_val_mask].to_numpy()
text_test_indices = model_df.index[text_test_mask].to_numpy()

print("Text cohort temporal split")
print("-" * 40)
print("Train (2007–2012):", len(text_train_indices))
print("Validation (2013):", len(text_val_indices))
print("Test (2014):", len(text_test_indices))


Text cohort temporal split
----------------------------------------
Train (2007–2012): 59350
Validation (2013): 48726
Test (2014): 15175


In [87]:
assert len(text_train_indices) == 59350
assert len(text_val_indices) == 48726
assert len(text_test_indices) == 15175

# Ensure temporal cohorts do not overlap
assert len(set(text_train_indices) & set(text_val_indices)) == 0
assert len(set(text_train_indices) & set(text_test_indices)) == 0
assert len(set(text_val_indices) & set(text_test_indices)) == 0

# Verify every selected observation has usable text
assert usable_desc_mask.loc[text_train_indices].all()
assert usable_desc_mask.loc[text_val_indices].all()
assert usable_desc_mask.loc[text_test_indices].all()

print("Text-cohort split validation passed.")


Text-cohort split validation passed.


## 11. Export Preprocessing Artefacts and Handoff

The preprocessing artefacts required by downstream notebooks are persisted under the Notebook 03 processed-data directory. Large structured matrices are not exported by default because they can be deterministically reconstructed from the saved transformations and source data, avoiding unnecessary duplication in Google Drive.

The exported artefacts include:

- training-derived numerical medians;
- the fitted categorical encoder;
- the ordered 90-feature structured feature-name list;
- the full structured temporal split definition;
- the matched text-cohort temporal split definition; and
- the train/validation/test row indices for the matched text-eligible cohort.


In [88]:
import joblib

NB3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame({
    "feature": train_numerical_medians.index,
    "training_median": train_numerical_medians.values
}).to_csv(
    NB3_OUTPUT_DIR / "training_numerical_medians.csv",
    index=False
)

pd.DataFrame({
    "feature": final_structured_feature_names
}).to_csv(
    NB3_OUTPUT_DIR / "final_structured_feature_names.csv",
    index=False
)

pd.DataFrame({
    "split": ["train", "validation", "test"],
    "start_year": [2007, 2017, 2018],
    "end_year": [2016, 2017, 2018]
}).to_csv(
    NB3_OUTPUT_DIR / "temporal_split_definition.csv",
    index=False
)

pd.DataFrame({
    "split": ["train", "validation", "test"],
    "start_year": [2007, 2013, 2014],
    "end_year": [2012, 2013, 2014],
    "text_required": [True, True, True],
    "rows": [
        len(text_train_indices),
        len(text_val_indices),
        len(text_test_indices)
    ]
}).to_csv(
    NB3_OUTPUT_DIR / "text_cohort_split_definition.csv",
    index=False
)

np.save(
    NB3_OUTPUT_DIR / "text_train_indices.npy",
    text_train_indices
)

np.save(
    NB3_OUTPUT_DIR / "text_val_indices.npy",
    text_val_indices
)

np.save(
    NB3_OUTPUT_DIR / "text_test_indices.npy",
    text_test_indices
)

joblib.dump(
    categorical_encoder,
    NB3_OUTPUT_DIR / "categorical_encoder.joblib"
)

print(
    "Notebook 03 preprocessing artefacts saved to:",
    NB3_OUTPUT_DIR
)


Notebook 03 preprocessing artefacts saved to: /content/drive/MyDrive/Credit_Risk_Thesis/data/processed/notebook_03


### 11.1 Export Model-Ready Structured Matrices

To maintain a clear separation between preprocessing and modelling, the final
structured train, validation, and test matrices produced in this notebook are
persisted for direct use by subsequent modelling notebooks.

The structured matrices are stored in sparse format to reduce storage and
memory requirements. Their corresponding binary target arrays are saved
separately.

This allows Notebook 04 to consume the exact preprocessing output generated
here without repeating feature engineering, imputation, or categorical
encoding.

In [89]:
from scipy import sparse

# Save final model-ready structured matrices
sparse.save_npz(
    NB3_OUTPUT_DIR / "X_train_structured.npz",
    X_train_final
)

sparse.save_npz(
    NB3_OUTPUT_DIR / "X_val_structured.npz",
    X_val_final
)

sparse.save_npz(
    NB3_OUTPUT_DIR / "X_test_structured.npz",
    X_test_final
)

# Save corresponding targets
np.save(
    NB3_OUTPUT_DIR / "y_train.npy",
    y_train.to_numpy()
)

np.save(
    NB3_OUTPUT_DIR / "y_val.npy",
    y_val.to_numpy()
)

np.save(
    NB3_OUTPUT_DIR / "y_test.npy",
    y_test.to_numpy()
)

print("Model-ready structured matrices and targets saved.")

Model-ready structured matrices and targets saved.


## 12. Notebook Summary and Handoff

### Key Findings

- The binary modelling population was restricted to resolved `Fully Paid` and `Charged Off` outcomes, producing **1,345,310 observations** with an overall default rate of approximately 19.96%.
- The 64 audited source features were separated into one textual modality (`desc`) and structured predictors.
- FICO range boundaries were consolidated into a single midpoint score, `earliest_cr_line` was transformed into `credit_history_months`, and `emp_length` was converted into ordered numerical tenure.
- The engineered structured set contains **57 numerical and 5 categorical predictors** before missingness-indicator expansion.
- Six event-recency missingness indicators were added because absence may itself carry information. Numerical imputation statistics and categorical encoding were learned using training data only.
- One-hot encoding expanded the five categorical predictors into 27 columns. The final structured representation contains **90 model-ready features**.
- The full structured experiment uses a chronological out-of-time split of **2007–2016 training, 2017 validation, and 2018 test**, containing 1,119,699, 169,300, and 56,311 observations respectively.
- Borrower-description availability was found to be strongly time-dependent. Usable `desc` records decline sharply over successive vintages, become effectively unavailable from 2015 onward, and are absent in the 2017–2018 structured validation/test periods.
- Because the full structured split cannot support text-based evaluation, a separate **matched text-eligible cohort** was defined: **2007–2012 training (59,350), 2013 validation (48,726), and 2014 test (15,175)**.
- Direct structured-only, text-only, and hybrid comparisons must use this matched text cohort so that performance differences reflect modelling representation rather than different borrower populations.
- Text-based findings are therefore conditional on description availability and should not be interpreted as representative of the entire Lending Club population.
- Global numerical scaling is intentionally deferred to model-specific pipelines and must be fitted on training data only.
- The final 90-feature structured train, validation, and test matrices and their
corresponding target arrays were persisted as modelling-ready artefacts,
allowing subsequent modelling notebooks to consume the preprocessing output
directly without repeating feature engineering or preprocessing.

### Handoff to Subsequent Notebooks

Notebook 04 consumes the persisted 90-feature structured train, validation, and test matrices and their corresponding target arrays to establish structured credit-risk baselines and evaluate out-of-time generalisation, without repeating the feature-engineering or preprocessing pipeline.
